# Detecção de Anomalias em Vídeo

Este notebook demonstra como implementar detecção de anomalias em vídeos usando OpenCV e subtração de fundo.

## Objetivo

Aprender a detectar objetos em movimento e anomalias em vídeos, aplicando técnicas de processamento de imagem e visão computacional.

## Aplicações Práticas

- Monitoramento de segurança em shopping centers
- Detecção de objetos caídos em armazéns
- Monitoramento de pacientes em hospitais
- Organização e segurança de estoques
- Análise de eventos em tempo real

In [ ]:
# Importação das bibliotecas necessárias

import cv2  # OpenCV para processamento de vídeo e imagens
import numpy as np  # NumPy para manipulação de arrays numéricos
from pathlib import Path  # Para manipulação de caminhos de arquivos
from IPython.display import HTML, Video  # Para exibir vídeos no notebook

print("✅ Bibliotecas importadas com sucesso!")
print(f"OpenCV versão: {cv2.__version__}")

## 1. Configuração do Vídeo de Entrada

Defina o caminho para o vídeo que será analisado. Você pode usar um vídeo local ou fazer upload de um arquivo.


In [ ]:
# Definir caminho do vídeo de entrada
# Substitua pelo caminho do seu vídeo
video_path = "video_entrada.mp4"  # Altere para o caminho do seu vídeo

# Verificar se o arquivo existe
video_file = Path(video_path)
if not video_file.exists():
    print(f"⚠️ Arquivo de vídeo não encontrado: {video_path}")
    print("📝 Por favor, defina o caminho correto do vídeo na variável 'video_path' acima.")
else:
    print(f"✅ Vídeo encontrado: {video_path}")
    print(f"📁 Tamanho do arquivo: {video_file.stat().st_size / (1024*1024):.2f} MB")

## 2. Configuração do Vídeo de Saída

Abrir o vídeo de entrada e configurar o vídeo de saída com as mesmas propriedades (FPS, resolução).


In [ ]:

# Abrir o vídeo de entrada
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"❌ Erro ao abrir o vídeo: {video_path}")

# Obter propriedades do vídeo
fps = int(cap.get(cv2.CAP_PROP_FPS))  # Frames por segundo
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  # Largura em pixels
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # Altura em pixels
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Total de frames

print(f"📹 Propriedades do vídeo:")
print(f"   - Resolução: {width}x{height} pixels")
print(f"   - FPS: {fps}")
print(f"   - Total de frames: {total_frames}")
print(f"   - Duração aproximada: {total_frames/fps:.2f} segundos")

# Configurar codec para o vídeo de saída
# MP4V é um codec popular para vídeos MP4
# Codec = Encoder (comprime) + Decoder (descomprime)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Criar objeto VideoWriter para salvar o vídeo processado
output_path = 'output_anomaly_detection.mp4'
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

if not out.isOpened():
    raise ValueError(f"❌ Erro ao criar arquivo de vídeo de saída: {output_path}")

print(f"✅ Vídeo de saída configurado: {output_path}")


## 3. Detecção de Anomalias usando Subtração de Fundo

A subtração de fundo é uma técnica que compara cada frame com um modelo do fundo estático para identificar objetos em movimento.

### Parâmetros do Detector de Fundo (MOG2):

- **history (500)**: Quantos frames anteriores usar para entender o fundo
  - Mais frames = modelo de fundo mais estável, mas processamento mais lento
  - Menos frames = modelo mais adaptável, mas pode incluir objetos em movimento no fundo

- **varThreshold (50)**: Sensibilidade para detectar mudanças
  - Valores menores = mais sensível (detecta movimentos menores, mas mais falsos positivos)
  - Valores maiores = menos sensível (detecta apenas movimentos significativos)

- **detectShadows (True)**: Se deve considerar sombras como movimento
  - True = detecta sombras (pode aumentar falsos positivos)
  - False = ignora sombras (pode perder objetos com sombras grandes)

In [ ]:

# Criar detector de fundo usando algoritmo MOG2 (Mixture of Gaussians)
# Este algoritmo aprende um modelo do fundo estático e identifica objetos em movimento
fgbg = cv2.createBackgroundSubtractorMOG2(
    history=500,        # Usa 500 frames anteriores para entender o fundo
    varThreshold=50,   # Sensibilidade: valores menores = mais sensível
    detectShadows=True # Detecta sombras como movimento
)

print("🔄 Processando vídeo frame por frame...")
print("⏳ Isso pode levar alguns minutos dependendo do tamanho do vídeo...\n")

frame_count = 0
anomalies_detected = 0

# Processar cada frame do vídeo
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_count += 1
    
    # Aplicar subtração de fundo
    # Retorna uma máscara binária: pixels brancos = movimento, pixels pretos = fundo
    fgmask = fgbg.apply(frame)
    
    # Remover ruídos pequenos usando morfologia matemática
    # MORPH_OPEN remove pequenos objetos (ruídos) mantendo apenas objetos grandes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    fgmask_cleaned = cv2.morphologyEx(fgmask, cv2.MORPH_OPEN, kernel)
    
    # Encontrar contornos dos objetos detectados
    # RETR_EXTERNAL: retorna apenas contornos externos (não aninhados)
    # CHAIN_APPROX_SIMPLE: comprime contornos, removendo pontos redundantes
    contours, _ = cv2.findContours(fgmask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Processar cada contorno encontrado
    for contour in contours:
        # Calcular área do contorno
        area = cv2.contourArea(contour)
        
        # Filtrar apenas objetos grandes o suficiente (limite mínimo de 500 pixels)
        # Isso evita detectar ruídos e pequenos movimentos irrelevantes
        if area > 500:
            anomalies_detected += 1
            
            # Obter retângulo delimitador do objeto
            x, y, w, h = cv2.boundingRect(contour)
            
            # Desenhar retângulo vermelho ao redor do objeto anômalo
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
            
            # Adicionar texto "ANOMALY DETECTED" acima do retângulo
            cv2.putText(frame, "ANOMALY DETECTED", (x, y - 10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    
    # Salvar frame processado no vídeo de saída
    out.write(frame)
    
    # Mostrar progresso a cada 30 frames
    if frame_count % 30 == 0:
        progress = (frame_count / total_frames) * 100
        print(f"📊 Progresso: {progress:.1f}% ({frame_count}/{total_frames} frames)")

# Liberar recursos
cap.release()
out.release()

print(f"\n✅ Processamento concluído!")
print(f"📹 Total de frames processados: {frame_count}")
print(f"🚨 Anomalias detectadas: {anomalies_detected}")
print(f"💾 Vídeo salvo em: {output_path}")

## 4. Exibir Vídeo Processado

Exibir o vídeo com as anomalias destacadas em vermelho.

In [ ]:

# Exibir o vídeo processado no notebook
if Path(output_path).exists():
    print(f"📹 Exibindo vídeo processado: {output_path}")
    print("🔴 Objetos anômalos aparecem destacados com retângulos vermelhos")
    print("📝 Use os controles do player para pausar, avançar ou retroceder\n")
    
    # Exibir vídeo usando IPython.display.Video (método mais simples)
    Video(output_path, width=640, height=480)
else:
    print(f"❌ Arquivo de vídeo não encontrado: {output_path}")
    print("📝 Certifique-se de que o processamento foi concluído com sucesso.")